<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>


# **Taxi Tip Prediction using Scikit-Learn and Snap ML**


Estimated time needed: **30** minutes


En esta sesión de ejercicios, consolidará sus habilidades de modelado de aprendizaje automático (ML) mediante el uso de un modelo de regresión popular: el árbol de decisiones. Utilizará un conjunto de datos reales para entrenar dicho modelo. El conjunto de datos incluye información sobre la propina de los taxis y fue recopilado y proporcionado a la Comisión de Taxis y Limusinas de la Ciudad de Nueva York (TLC) por proveedores de tecnología autorizados según los Programas de Mejora de Pasajeros de Taxis y Limusinas (TPEP/LPEP). Utilizará el modelo entrenado para predecir el monto de la propina pagada.

En la sesión de ejercicios actual, practicará no solo la interfaz Python de Scikit-Learn, sino también la API Python que ofrece la biblioteca Snap Machine Learning (Snap ML). Snap ML es una biblioteca IBM de alto rendimiento para el modelado ML. Proporciona implementaciones de CPU/GPU altamente eficientes de modelos lineales y modelos basados ​​en árboles. Snap ML no solo acelera los algoritmos ML a través del conocimiento del sistema, sino que también ofrece algoritmos ML novedosos con la mejor precisión de su clase. Para obtener más información, visite la página de información de [snapml](https://ibm.biz/BdPfxy).


## Objectives


After completing this lab you will be able to:


* Perform basic data preprocessing using Scikit-Learn
* Model a regression task using the Scikit-Learn and Snap ML Python APIs
* Train a Decision Tree Regressor model using Scikit-Learn and Snap ML
* Run inference and assess the quality of the trained models


## Table of Contents


<div class="alert alert-block alert-info" style="margin-top: 10px">
    <ol>
        <li><a href="#introduction">Introduction</a></li>
        <li><a href="#import_libraries">Import Libraries</a></li>
        <li><a href="#dataset_analysis">Dataset Analysis</a></li>
        <li><a href="#dataset_preprocessing">Dataset Preprocessing</a></li>
        <li><a href="#dataset_split">Dataset Train/Test Split</a></li>
        <li><a href="#dt_sklearn">Build a Decision Tree Regressor model with Scikit-Learn</a></li>
        <li><a href="#dt_snap">Build a Decision Tree Regressor model with Snap ML</a></li>
        <li><a href="#dt_sklearn_snap">Evaluate the Scikit-Learn and Snap ML Decision Tree Regressors</a></li>
    </ol>
</div>
<br>
<hr>


<div id="Introducción">
<h2>Introducción</h2>
<br>El conjunto de datos utilizado en esta sesión de ejercicios está disponible públicamente aquí: https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page (todos los derechos reservados por Taxi & Limousine Commission(TLC), City of New York). En este cuaderno se utilizan los registros de viajes de taxis amarillos de TLC de junio de 2019. La predicción del monto de la propina se puede modelar como un problema de regresión. Para entrenar el modelo, puede utilizar parte del conjunto de datos de entrada y los datos restantes se pueden utilizar para evaluar la calidad del modelo entrenado. Primero, descarguemos el conjunto de datos.
<br>
</div>

In [ ]:
# download June 2020 TLC Yellow Taxi Trip records
# !wget -nc https://s3.amazonaws.com/nyc-tlc/trip+data/yellow_tripdata_2019-06.csv
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/yellow_tripdata_2019-06.csv

¿Sabías que...? Cuando se trata de aprendizaje automático, lo más probable es que trabajes con grandes conjuntos de datos. Como empresa, ¿dónde puedes alojar tus datos? IBM ofrece una oportunidad única para las empresas, con 10 TB de IBM Cloud Object Storage: [Regístrate ahora gratis](https://ibm.biz/BdPfxf)


<div id="import_libraries">
    <h2>Import Libraries</h2>
</div>


In [ ]:
# Snap ML is available on PyPI. To install it simply run the pip command below.
!pip install snapml
!pip install scikit-learn
!pip install matplotlib
!pip install pandas 
!pip install numpy 
%matplotlib inline

In [ ]:
# Import the libraries we need to use in this lab
from __future__ import print_function
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, StandardScaler, MinMaxScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import mean_squared_error
import time
import warnings
import gc, sys
warnings.filterwarnings('ignore')

<div id="dataset_analysis">
    <h2>Dataset Analysis</h2>
</div>


En esta sección, leerá el conjunto de datos en un marco de datos de Pandas y visualizará su contenido. También verá algunas estadísticas de datos.

Nota: Un marco de datos de Pandas es una estructura de datos tabular bidimensional, de tamaño variable y potencialmente heterogénea. Para obtener más información: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html.


In [ ]:
# read the input data
raw_data = pd.read_csv('yellow_tripdata_2019-06.csv')
print("There are " + str(len(raw_data)) + " observations in the dataset.")
print("There are " + str(len(raw_data.columns)) + " variables in the dataset.")

# display first rows in the dataset
raw_data.head()

In [ ]:
#Reducing the data size to 100000 records
raw_data=raw_data.head(100000)

Cada fila del conjunto de datos representa un viaje en taxi. Como se muestra arriba, cada fila tiene 18 variables. Una variable se llama tip_amount y representa la variable objetivo. Tu objetivo será entrenar un modelo que use las otras variables para predecir el valor de la variable tip_amount. Primero, limpiemos el conjunto de datos y recuperemos estadísticas básicas sobre la variable objetivo.

In [ ]:
# Algunos viajes no dejan propina. Se supone que estas propinas se pagaron en efectivo.
# Para este estudio, omitimos todas estas filas
raw_data = raw_data[raw_data['tip_amount'] > 0]

# También eliminamos algunos valores atípicos, es decir, aquellos en los que la propina fue mayor que el costo de la tarifa.
raw_data = raw_data[(raw_data['tip_amount'] <= raw_data['fare_amount'])]

# eliminamos viajes con costos de tarifa muy altos
raw_data = raw_data[((raw_data['fare_amount'] >=2) & (raw_data['fare_amount'] < 200))]

# eliminamos las variables que incluyen la variable de destino, es decir, total_amount
clean_data = raw_data.drop(['total_amount'], axis=1)

# Liberamos la memoria ocupada por raw_data ya que ya no la necesitamos
# Estamos trabajando con un conjunto de datos grande, por lo que debemos asegurarnos de no quedarnos sin memoria
del raw_data
gc.collect()

# imprime el número de viajes que quedan en el conjunto de datos
print("There are " + str(len(clean_data)) + " observations in the dataset.")
print("There are " + str(len(clean_data.columns)) + " variables in the dataset.")

plt.hist(clean_data.tip_amount.values, 16, histtype='bar', facecolor='g')
plt.show()

print("Minimum amount value is ", np.min(clean_data.tip_amount.values))
print("Maximum amount value is ", np.max(clean_data.tip_amount.values))
print("90% of the trips have a tip amount less or equal than ", np.percentile(clean_data.tip_amount.values, 90))

In [ ]:
# display first rows in the dataset
clean_data.head()

Si observamos el conjunto de datos con más detalle, veremos que contiene información como las fechas y horas de recogida y entrega, los lugares de recogida y entrega, los tipos de pago, el número de pasajeros informados por el conductor, etc. Antes de entrenar un modelo de ML, necesitaremos preprocesar los datos. Necesitamos transformar los datos en un formato que los modelos puedan manejar correctamente. Por ejemplo, necesitamos codificar las características categóricas.


<div id="dataset_preprocessing">
    <h2>Dataset Preprocessing</h2>
</div>


In this subsection you will prepare the data for training. 


In [ ]:
# convertir a fecha y hora
clean_data['tpep_dropoff_datetime'] = pd.to_datetime(clean_data['tpep_dropoff_datetime'])
clean_data['tpep_pickup_datetime'] = pd.to_datetime(clean_data['tpep_pickup_datetime'])

#extraer hora de recogida y entrega
clean_data['pickup_hour'] = clean_data['tpep_pickup_datetime'].dt.hour
clean_data['dropoff_hour'] = clean_data['tpep_dropoff_datetime'].dt.hour

# Extraer el día de recogida y entrega de la semana
clean_data['pickup_day'] = clean_data['tpep_pickup_datetime'].dt.weekday
clean_data['dropoff_day'] = clean_data['tpep_dropoff_datetime'].dt.weekday

# Calcular el tiempo de viaje en minutos
clean_data['trip_time'] = (clean_data['tpep_dropoff_datetime'] - clean_data['tpep_pickup_datetime']).dt.total_seconds() / 60

# reducir el tamaño del conjunto de datos si es necesario
first_n_rows = 1000000
clean_data = clean_data.head(first_n_rows)

In [ ]:
# indicar las fechas y horas de recogida y entrega
clean_data = clean_data.drop(['tpep_pickup_datetime', 'tpep_dropoff_datetime'], axis=1)

# Algunas características son categóricas, necesitamos codificarlas
# Para codificarlas, usamos la codificación one-hot del paquete Pandas
get_dummy_col = ["VendorID","RatecodeID","store_and_fwd_flag","PULocationID", "DOLocationID","payment_type", "pickup_hour", "dropoff_hour", "pickup_day", "dropoff_day"]
proc_data = pd.get_dummies(clean_data, columns = get_dummy_col)

# Liberamos la memoria ocupada por clean_data ya que ya no la necesitamos
# Estamos trabajando con un conjunto de datos grande, por lo que debemos asegurarnos de no quedarnos sin memoria
del clean_data
gc.collect()

In [ ]:
# extract the labels from the dataframe
y = proc_data[['tip_amount']].values.astype('float32')

# elimina la variable de destino de la matriz de características
proc_data = proc_data.drop(['tip_amount'], axis=1)

# obtener la matriz de características utilizada para el entrenamiento
X = proc_data.values

# normalizar la matriz de características
X = normalize(X, axis=1, norm='l1', copy=False)

# imprime la forma de la matriz de características y el vector de etiquetas
print('X.shape=', X.shape, 'y.shape=', y.shape)

<div id="dataset_split">
    <h2>Dataset Train/Test Split</h2>
</div>


Ahora que el conjunto de datos está listo para construir los modelos de clasificación, primero debe dividir el conjunto de datos preprocesados ​​en un subconjunto que se utilizará para entrenar el modelo (el conjunto de entrenamiento) y un subconjunto que se utilizará para evaluar la calidad del modelo (el conjunto de prueba).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('X_train.shape=', X_train.shape, 'Y_train.shape=', y_train.shape)
print('X_test.shape=', X_test.shape, 'Y_test.shape=', y_test.shape)

<div id="dt_sklearn">
    <h2>Build a Decision Tree Regressor model with Scikit-Learn</h2>
    <h2>Construya un modelo de regresor de árbol de decisión con Scikit-Learn</h2>
</div>


In [ ]:
# import the Decision Tree Regression Model from scikit-learn
from sklearn.tree import DecisionTreeRegressor

# para obtener una salida reproducible en múltiples llamadas de función, establezca random_state en un valor entero determinado
sklearn_dt = DecisionTreeRegressor(max_depth=8, random_state=35)

# Entrene la Decision Tree Regressor usando scikit-learn
t0 = time.time()
sklearn_dt.fit(X_train, y_train)
sklearn_time = time.time()-t0
print("[Scikit-Learn] Training time (s):  {0:.5f}".format(sklearn_time))

<div id="dt_snapml">
    <h2>Build a Decision Tree Regressor model with Snap ML</h2>
    <h2>Construya un modelo de regresor de árbol de decisión con Snap ML</h2>
</div>


In [ ]:
# import the Decision Tree Regressor Model from Snap ML
from snapml import DecisionTreeRegressor

# a diferencia del árbol de decisiones de sklearn, Snap ML ofrece entrenamiento de CPU/GPU multiproceso
# para usar la GPU, es necesario configurar el parámetro use_gpu en True
# snapml_dt = DecisionTreeRegressor(max_depth=4, random_state=45, use_gpu=True)

# para establecer la cantidad de subprocesos de CPU utilizados en el momento del entrenamiento, es necesario establecer el parámetro n_jobs
# para obtener una salida reproducible en múltiples llamadas de función, establezca random_state en un valor entero determinado
snapml_dt = DecisionTreeRegressor(max_depth=8, random_state=45, n_jobs=4)

# train a Decision Tree Regressor model using Snap ML
t0 = time.time()
snapml_dt.fit(X_train, y_train)
snapml_time = time.time()-t0
print("[Snap ML] Training time (s):  {0:.5f}".format(snapml_time))

<div id="dt_sklearn_snapml">
    <h2>Evaluate the Scikit-Learn and Snap ML Decision Tree Regressor Models</h2>
    <h2>Evaluar los modelos de regresores de árboles de decisión de Scikit-Learn y Snap ML</h2>
</div>


In [ ]:
# Snap ML vs Scikit-Learn training speedup
training_speedup = sklearn_time/snapml_time
print('[Decision Tree Regressor] Snap ML vs. Scikit-Learn speedup : {0:.2f}x '.format(training_speedup))

# ejecutar la predicción usando el modelo sklearn
sklearn_pred = sklearn_dt.predict(X_test)

# evaluar el error cuadrático medio en el conjunto de datos de prueba
sklearn_mse = mean_squared_error(y_test, sklearn_pred)
print('[Scikit-Learn] MSE score : {0:.3f}'.format(sklearn_mse))

# ejecutar predicción usando el modelo Snap ML
snapml_pred = snapml_dt.predict(X_test)

# evaluar el error cuadrático medio en el conjunto de datos de prueba
snapml_mse = mean_squared_error(y_test, snapml_pred)
print('[Snap ML] MSE score : {0:.3f}'.format(snapml_mse))

Como se muestra arriba, ambos modelos de árboles de decisión proporcionan la misma puntuación en el conjunto de datos de prueba. Sin embargo, Snap ML ejecuta la rutina de entrenamiento más rápido que Scikit-Learn. Esta es una de las ventajas de usar Snap ML: la aceleración del entrenamiento de los modelos de aprendizaje automático clásicos, como los modelos lineales y basados ​​en árboles. Para obtener más ejemplos de Snap ML, visite [snapml-examples](https://ibm.biz/BdPfxP). Además, como se muestra arriba, Snap ML no solo acelera sin problemas las aplicaciones de scikit-learn, sino que la API de Python de la biblioteca también es compatible con las métricas y los preprocesadores de datos de scikit-learn.


## Authors


Andreea Anghel


### Other Contributors


Sangeeth Keeriyadath 

Joseph Santarcangelo


<!-- ## Change Log --!>


<!-- |  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2021-08-31  | 0.1  | AAN  |  Created Lab Content |
| 2023-01-24  | 0.2  | Anita Verma  |  Reduced data size| --!>


 Copyright &copy; 2021 IBM Corporation. This notebook and its source code are released under the terms of the [MIT License](https://cognitiveclass.ai/mit-license/).
